In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 11


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.609221749007702
Epoch 2/100, Loss: 2.4459606632590294
Epoch 3/100, Loss: 2.4771324545145035
Epoch 4/100, Loss: 2.5794943645596504
Epoch 5/100, Loss: 2.5366047620773315
Epoch 6/100, Loss: 2.494461126625538
Epoch 7/100, Loss: 2.3925967812538147
Epoch 8/100, Loss: 2.7009518444538116
Epoch 9/100, Loss: 2.4879960492253304
Epoch 10/100, Loss: 2.457835301756859
Epoch 11/100, Loss: 2.3445565178990364
Epoch 12/100, Loss: 2.5918062180280685
Epoch 13/100, Loss: 2.4315365739166737
Epoch 14/100, Loss: 2.5754426643252373
Epoch 15/100, Loss: 3.0957513004541397
Epoch 16/100, Loss: 2.631595730781555
Epoch 17/100, Loss: 2.7033082246780396


Epoch 18/100, Loss: 2.4927254393696785
Epoch 19/100, Loss: 2.2310572415590286
Epoch 20/100, Loss: 2.4485263526439667
Epoch 21/100, Loss: 2.5679445937275887
Epoch 22/100, Loss: 2.3504332825541496
Epoch 23/100, Loss: 2.494469068944454
Epoch 24/100, Loss: 2.5504394099116325
Epoch 25/100, Loss: 2.6893958896398544
Epoch 26/100, Loss: 2.4947719126939774
Epoch 27/100, Loss: 2.5574314519762993
Epoch 28/100, Loss: 2.4267420172691345
Epoch 29/100, Loss: 2.4964586719870567
Epoch 30/100, Loss: 2.5154936462640762
Epoch 31/100, Loss: 2.441619075834751
Epoch 32/100, Loss: 2.381852760910988
Epoch 33/100, Loss: 2.564363531768322
Epoch 34/100, Loss: 2.3853904083371162


Epoch 35/100, Loss: 2.5144168958067894
Epoch 36/100, Loss: 2.593670703470707
Epoch 37/100, Loss: 2.647905595600605
Epoch 38/100, Loss: 2.5775167047977448
Epoch 39/100, Loss: 2.5770293846726418
Epoch 40/100, Loss: 2.489797279238701
Epoch 41/100, Loss: 2.65754321962595
Epoch 42/100, Loss: 2.515278458595276
Epoch 43/100, Loss: 2.59809809923172
Epoch 44/100, Loss: 2.361557260155678
Epoch 45/100, Loss: 2.586212210357189
Epoch 46/100, Loss: 2.4099755361676216
Epoch 47/100, Loss: 2.5647121742367744
Epoch 48/100, Loss: 2.6343067958950996
Epoch 49/100, Loss: 2.4433158338069916
Epoch 50/100, Loss: 2.819645047187805
Epoch 51/100, Loss: 2.4750514775514603


Epoch 52/100, Loss: 2.5308874621987343
Epoch 53/100, Loss: 2.408222980797291
Epoch 54/100, Loss: 2.466579094529152
Epoch 55/100, Loss: 2.2522589191794395
Epoch 56/100, Loss: 2.496432699263096
Epoch 57/100, Loss: 3.1716118454933167
Epoch 58/100, Loss: 2.602905608713627
Epoch 59/100, Loss: 2.5436235517263412
Epoch 60/100, Loss: 2.5401155948638916
Epoch 61/100, Loss: 2.6101967245340347
Epoch 62/100, Loss: 2.333831164985895
Epoch 63/100, Loss: 2.597722090780735
Epoch 64/100, Loss: 2.5410186797380447
Epoch 65/100, Loss: 2.515992186963558
Epoch 66/100, Loss: 2.47756227850914
Epoch 67/100, Loss: 2.415529102087021
Epoch 68/100, Loss: 2.4268108010292053


Epoch 69/100, Loss: 2.474801667034626
Epoch 70/100, Loss: 2.6625565513968468
Epoch 71/100, Loss: 2.4976409897208214
Epoch 72/100, Loss: 2.4844680726528168
Epoch 73/100, Loss: 2.7773054242134094
Epoch 74/100, Loss: 2.405972495675087
Epoch 75/100, Loss: 2.473934903740883
Epoch 76/100, Loss: 2.473983481526375
Epoch 77/100, Loss: 2.658280149102211
Epoch 78/100, Loss: 2.5138888135552406
Epoch 79/100, Loss: 2.5689857229590416
Epoch 80/100, Loss: 2.3460301533341408
Epoch 81/100, Loss: 2.594384215772152
Epoch 82/100, Loss: 2.386183299124241
Epoch 83/100, Loss: 2.369439549744129
Epoch 84/100, Loss: 2.5794462338089943
Epoch 85/100, Loss: 2.4723986387252808


...

Epoch 52/100, Loss: 2.148788668215275
Epoch 53/100, Loss: 1.9551063179969788
Epoch 54/100, Loss: 1.8836794383823872
Epoch 55/100, Loss: 1.8189442940056324
Epoch 56/100, Loss: 1.8721905946731567
Epoch 57/100, Loss: 1.865684948861599
Epoch 58/100, Loss: 2.2295323349535465
Epoch 59/100, Loss: 1.858375422656536
Epoch 60/100, Loss: 2.238897319883108
Epoch 61/100, Loss: 2.372430719435215
Epoch 62/100, Loss: 1.8878282234072685
Epoch 63/100, Loss: 1.7326361387968063


Epoch 64/100, Loss: 2.1653399541974068
Epoch 65/100, Loss: 1.888366211205721
Epoch 66/100, Loss: 1.9545510970056057
Epoch 67/100, Loss: 1.988158904016018
Epoch 68/100, Loss: 1.9093416295945644
Epoch 69/100, Loss: 1.8941664770245552
Epoch 70/100, Loss: 2.427762895822525
Epoch 71/100, Loss: 1.5756388194859028
Epoch 72/100, Loss: 2.013338875025511
Epoch 73/100, Loss: 1.5188488103449345
Epoch 74/100, Loss: 1.8430057354271412
Epoch 75/100, Loss: 1.83644225820899
Epoch 76/100, Loss: 2.9384645894169807


Epoch 77/100, Loss: 1.6776548996567726
Epoch 78/100, Loss: 1.8926027715206146
Epoch 79/100, Loss: 1.9187960997223854
Epoch 80/100, Loss: 1.678288571536541
Epoch 81/100, Loss: 2.013811245560646
Epoch 82/100, Loss: 2.015066225081682
Epoch 83/100, Loss: 2.0419316589832306
Epoch 84/100, Loss: 1.8133759945631027
Epoch 85/100, Loss: 2.0585696138441563
Epoch 86/100, Loss: 2.5611723884940147
Epoch 87/100, Loss: 2.1674607768654823
Epoch 88/100, Loss: 2.0931705124676228
Epoch 89/100, Loss: 2.083685837686062


Epoch 90/100, Loss: 2.108265731483698
Epoch 91/100, Loss: 1.7492974884808064
Epoch 92/100, Loss: 2.0146694630384445
Epoch 93/100, Loss: 1.6148098707199097
Epoch 94/100, Loss: 1.768626183271408
Epoch 95/100, Loss: 1.74943857640028
Epoch 96/100, Loss: 1.6133918277919292
Epoch 97/100, Loss: 1.530499018728733
Epoch 98/100, Loss: 2.2490188144147396
Epoch 99/100, Loss: 2.0003980584442616


Epoch 100/100, Loss: 1.911668837070465
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6455
